In [1]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

def criar_lstm_otimizadora(look_back, num_features, num_ativos):
    modelo = Sequential()
    modelo.add(LSTM(64, return_sequences=True, input_shape=(look_back, num_features)))
    modelo.add(Dropout(0.2))
    modelo.add(LSTM(32))
    
    # A CAMADA MÁGICA: N neurónios (um para cada ativo), ativados por Softmax
    modelo.add(Dense(num_ativos, activation='softmax')) 
    return modelo

In [2]:
import tensorflow as tf
import tensorflow.keras.backend as K

def max_sharpe_loss(y_true, y_pred):
    """
    y_true: Retornos reais futuros (Batch Size, Num_Ativos)
    y_pred: Pesos gerados pela rede (Batch Size, Num_Ativos)
    """
    # 1. Calcula o retorno diário da carteira: Somatório(Peso * Retorno)
    # A multiplicação é element-wise e depois somamos no eixo dos ativos (axis=-1)
    retorno_carteira = tf.reduce_sum(y_true * y_pred, axis=-1)
    
    # 2. Calcula a Esperança (Média dos retornos da carteira no batch)
    media_retorno = tf.reduce_mean(retorno_carteira)
    
    # 3. Calcula a Volatilidade (Desvio padrão da carteira no batch)
    volatilidade = tf.math.reduce_std(retorno_carteira)
    
    # 4. Calcula o Sharpe (adicionamos epsilon na divisão para evitar erro matemático de divisão por zero)
    # Nota: Assumimos taxa risk-free = 0 para a simplificação do gradiente.
    sharpe = media_retorno / (volatilidade + K.epsilon())
    
    # 5. Retornamos o Sharpe Negativo (A rede minimiza isto, logo maximiza o Sharpe real)
    return -sharpe

In [3]:
def min_variancia_loss(y_true, y_pred):
    retorno_carteira = tf.reduce_sum(y_true * y_pred, axis=-1)
    variancia = tf.math.reduce_variance(retorno_carteira)
    
    # Como já queremos minimizar o risco, retornamos a variância positiva
    return variancia

In [5]:
# Instanciar o modelo para, por exemplo, 50 ações
modelo_gestor = criar_lstm_otimizadora(look_back=60, num_features=6, num_ativos=50)

# Compilar com a Loss de Máximo Sharpe
modelo_gestor.compile(optimizer='adam', loss=max_sharpe_loss)

# TREINO DA IA
# X_treino: A tua matriz 3D histórica (Janelas de 60 dias com os Fatores FF5)
# Y_treino: A matriz 2D com os RETORNOS REAIS do dia 61 de todas as 50 ações!
modelo_gestor.fit(X_treino, Y_treino, epochs=50, batch_size=64)

NameError: name 'X_treino' is not defined

In [6]:
def max_sharpe_loss_institucional(y_true, y_pred):
    retorno_carteira = tf.reduce_sum(y_true * y_pred, axis=-1)
    sharpe = tf.reduce_mean(retorno_carteira) / (tf.math.reduce_std(retorno_carteira) + K.epsilon())
    
    # PENALIZAÇÃO DE CONCENTRAÇÃO (Herfindahl-Hirschman Index - HHI)
    # Somamos o quadrado dos pesos. Se um peso for 1.0 (100%), a penalidade é máxima.
    penalidade_concentracao = tf.reduce_sum(tf.square(y_pred), axis=-1)
    lambda_reg = 0.05 # Força da multa
    
    # A rede agora tenta maximizar o Sharpe, mas é multada se não diversificar!
    return -sharpe + (lambda_reg * tf.reduce_mean(penalidade_concentracao))